**1 - Install dependencies**


In [ ]:
!pip install ultralytics mlflow roboflow -q

**Cell 2: Download dataset from Roboflow**

In [ ]:
from roboflow import Roboflow

rf = Roboflow(api_key="K6R76fKleLzaSUeLNLRl")  # free account at app.roboflow.com
project = rf.workspace("david-lee-d0rhs").project("american-sign-language-letters")
dataset = project.version(1).download("yolov8", location="/content/data/raw")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to /content/data/raw in yolov8:: 100%|██████████| 3468/3468 [00:01<00:00, 2790.49it/s]


Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


**Cell 3: Fix data.yaml paths**

In [ ]:
import yaml

yaml_path = "/content/data/raw/data.yaml"
with open(yaml_path, "r") as f:
    config = yaml.safe_load(f)

config["train"] = "/content/data/raw/train/images"
config["val"]   = "/content/data/raw/valid/images"
config["test"]  = "/content/data/raw/test/images"

with open(yaml_path, "w") as f:
    yaml.dump(config, f)

print("data.yaml fixed:")
print(open(yaml_path).read())

data.yaml fixed:
names:
- A
- B
- C
- D
- E
- F
- G
- H
- I
- J
- K
- L
- M
- N
- O
- P
- Q
- R
- S
- T
- U
- V
- W
- X
- Y
- Z
nc: 26
roboflow:
  license: Public Domain
  project: american-sign-language-letters
  url: https://universe.roboflow.com/david-lee-d0rhs/american-sign-language-letters/dataset/1
  version: 1
  workspace: david-lee-d0rhs
test: /content/data/raw/test/images
train: /content/data/raw/train/images
val: /content/data/raw/valid/images



**Cell 4: Train**

In [ ]:
import os
os.environ["MLFLOW_EXPERIMENT_NAME"] = "sign-language"

# Enable MLflow tracking via settings (correct way in new Ultralytics)
!yolo settings mlflow=True

!yolo detect train \
  data=/content/data/raw/data.yaml \
  model=yolov8n.pt \
  imgsz=416 \
  epochs=50 \
  batch=16 \
  project=/content/runs \
  name=e1_yolov8n

✅ Updated 'mlflow=True'
JSONDict("/root/.config/Ultralytics/settings.json"):
{
  "settings_version": "0.0.8",
  "datasets_dir": "/content/datasets",
  "weights_dir": "weights",
  "runs_dir": "runs",
  "uuid": "a3d8d1371c620a7e347b2b1d97afdeaeddd19cbd13ebd19e6fb99e4357305253",
  "sync": true,
  "api_key": "",
  "openai_api_key": "",
  "clearml": true,
  "comet": true,
  "dvc": true,
  "mlflow": true,
  "raytune": true,
  "tensorboard": false,
  "wandb": false,
  "vscode_msg": true,
  "openvino_msg": true
}
💡 Learn more about Ultralytics Settings at https://docs.ultralytics.com/usage/settings
Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cut

**Cell 5 — Validate on test set:**

In [ ]:
!yolo val \
  model=/content/runs/e1_yolov8n/weights/best.pt \
  data=/content/data/raw/data.yaml \
  split=test

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 73 layers, 3,138,782 parameters, 0 gradients, 8.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 691.1±181.5 MB/s, size: 13.5 KB)
val: Scanning /content/data/raw/test/labels... 72 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 72/72 2.1Kit/s 0.0s
val: New cache created: /content/data/raw/test/labels.cache
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 5/5 3.4it/s 1.5s
                   all         72         72      0.885      0.894      0.939      0.758
                     A          1          1      0.221          1      0.995      0.895
                     B          3          3      0.915          1      0.995      0.813
                     C          4          4      0.916          1      0.995      0.846
                     D          1          1        0.9          1      0.995      0.796
 

**Cell 6 — Export to ONNX:**

In [ ]:
!yolo export \
  model=/content/runs/e1_yolov8n/weights/best.pt \
  format=onnx \
  imgsz=416 \
  dynamic=False

Ultralytics 8.4.215 🚀 Python-3.13.16 torch-2.14.1+cu130 CPU (Intel Xeon CPU @ 2.00GHz)
💡 ProTip: Export to OpenVINO format for best performance on Intel hardware. Learn more at https://docs.ultralytics.com/integrations/openvino
Model summary (fused): 73 layers, 3,138,782 parameters, 0 gradients, 3.7 GFLOPs

PyTorch: starting from '/content/runs/e1_yolov8n/weights/best.pt' with input shape (1, 3, 416, 416) BCHW and output shape(s) (1, 30, 3549) (6.2 MB)
requirements: Ultralytics requirements ['onnx>=1.12.0,<2.0.0', 'onnxruntime', 'onnxslim>=0.1.82'] not found, attempting AutoUpdate...
Using Python 3.13.16 environment at: /usr
Resolved 12 packages in 267ms
Prepared 4 packages in 554ms
Installed 4 packages in 47ms
 + colorama==0.4.6
 + onnx==1.23.2
 + onnxruntime==1.32.0
 + onnxslim==0.1.99

requirements: AutoUpdate success ✅ 1.5s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect


ONNX: starting export with onnx 1.23.2 opset 18...
ONNX: slimming with on

**Cell — Download everything before session ends:**

In [ ]:
'''import shutil
from google.colab import files

shutil.make_archive("/content/results_e1", "zip", "/content/runs/e1_yolov8n")
files.download("/content/results_e1.zip")
files.download("/content/runs/e1_yolov8n/weights/best.pt")
files.download("/content/runs/e1_yolov8n/weights/best.onnx")'''

'import shutil\nfrom google.colab import files\n\nshutil.make_archive("/content/results_e1", "zip", "/content/runs/e1_yolov8n")\nfiles.download("/content/results_e1.zip")\nfiles.download("/content/runs/e1_yolov8n/weights/best.pt")\nfiles.download("/content/runs/e1_yolov8n/weights/best.onnx")'

In [ ]:
# Download the mlflow database from Colab
from google.colab import files
files.download("/content/mlflow.db")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
import mlflow
print(mlflow.__version__)

3.18.0
